# ML 04 — Grouped split strategy and preprocessing

This notebook creates and inspects cross-validation folds directly from the modeling dataset. The primary evaluation holds out complete **plan families** so sibling layouts and wind directions from a family cannot leak into training. A layout-level split is shown only as a secondary interpolation comparison.

In [ ]:
from pathlib import Path
import os

# Works whether Jupyter starts in the repository root or this notebook's folder.
_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "data" / "raw").exists() and (p / "Machine Learning Pipeline").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root. Open this notebook from inside the T006 project folder.")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

processed_path = DATA_PROCESSED / "layout_level_dataset.csv"
if not processed_path.exists():
    raise FileNotFoundError("Run ML_01_dataset_understanding_data_dictionary.ipynb first; it builds this processed dataset from raw data.")
df = pd.read_csv(processed_path)
TARGET = "target_U_canopy_mean_m_s"
NUMERIC_FEATURES = ["plan_area_density", "plan_area_density_sq", "height_std_m", "height_max_m", "height_min_m", "height_range_m", "height_skew", "frac_taller_than_mean", "wind_direction_deg"]
CATEGORICAL_FEATURES = ["configuration", "horizontal_configuration", "density_class"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
BASELINE_NUMERIC = ["plan_area_density", "height_mean_m", "height_std_m", "height_max_m", "wind_direction_deg", "building_count"]
BASELINE_FEATURES = BASELINE_NUMERIC + CATEGORICAL_FEATURES
print("Dataset shape:", df.shape)
print("Target:", TARGET)

## 1. Create family-held-out folds

In [ ]:
from sklearn.model_selection import GroupKFold

X = df[FEATURES]
y = df[TARGET]
groups = df["family_id"]
cv = GroupKFold(n_splits=5)
fold_rows = []
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])
    assert train_groups.isdisjoint(test_groups)
    assert set(df.iloc[train_idx]["layout_id"]).isdisjoint(set(df.iloc[test_idx]["layout_id"]))
    fold_rows.append({
        "protocol": "family_5fold", "fold": fold,
        "train_cases": len(train_idx), "test_cases": len(test_idx),
        "train_families": len(train_groups), "test_families": len(test_groups),
        "train_family_ids": ",".join(sorted(train_groups)),
        "test_family_ids": ",".join(sorted(test_groups)),
    })
fold_table = pd.DataFrame(fold_rows)
display(fold_table.drop(columns=["train_family_ids", "test_family_ids"]))
print("Family leakage check passed for every fold.")

## 2. Compare with layout-held-out folds

In [ ]:
layout_cv = GroupKFold(n_splits=5)
layout_rows = []
for fold, (train_idx, test_idx) in enumerate(layout_cv.split(X, y, df["layout_id"]), start=1):
    train_layouts = set(df.iloc[train_idx]["layout_id"])
    test_layouts = set(df.iloc[test_idx]["layout_id"])
    assert train_layouts.isdisjoint(test_layouts)
    shared_families = set(df.iloc[train_idx]["family_id"]) & set(df.iloc[test_idx]["family_id"])
    layout_rows.append({"fold": fold, "train_cases": len(train_idx), "test_cases": len(test_idx), "shared_families_across_train_test": len(shared_families)})
display(pd.DataFrame(layout_rows))
print("Layout split can put related layout families on both sides. Treat it as a secondary interpolation protocol, not the primary generalization estimate.")

## 3. Demonstrate preprocessing without fitting on test data

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])
train_idx, test_idx = next(iter(cv.split(X, y, groups)))
X_train_transformed = preprocessor.fit_transform(X.iloc[train_idx])
X_test_transformed = preprocessor.transform(X.iloc[test_idx])
print("Train transformed shape:", X_train_transformed.shape)
print("Test transformed shape:", X_test_transformed.shape)
print("Preprocessor fit on training rows only; test rows were transformed without fitting.")

## 4. Save the fold manifest

In [ ]:
fold_path = RESULTS / "family_fold_manifest.csv"
fold_table.to_csv(fold_path, index=False)
print("Created:", fold_path)